In [1]:
import pyspark
from pyspark.sql import SparkSession

# 1. Start the Spark Engine for this new notebook
spark = SparkSession.builder.appName("Master_EDA").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# 2. Load the RAW DataFrames into memory
df_attr = spark.read.csv("data/features_attributes.csv", header=True, inferSchema=True)
df_fin = spark.read.csv("data/features_financials.csv", header=True, inferSchema=True)
df_click = spark.read.csv("data/feature_clickstream.csv", header=True, inferSchema=True)

print("Data loaded successfully! You can now run Phase 1.")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/16 07:40:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

Data loaded successfully! You can now run Phase 1.


In [7]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, length, trim

print("=== PHASE 1: FOUNDATION & KEY FORENSICS ===")

# 1. Duplicates Check (Rows vs Unique IDs)
click_tot, click_uniq = df_click.count(), df_click.select("Customer_ID").distinct().count()
fin_tot, fin_uniq = df_fin.count(), df_fin.select("Customer_ID").distinct().count()
attr_tot, attr_uniq = df_attr.count(), df_attr.select("Customer_ID").distinct().count()

print(f"Clickstream: {click_tot} Rows | {click_uniq} Unique IDs")
print(f"Financials:  {fin_tot} Rows | {fin_uniq} Unique IDs")
print(f"Attributes:  {attr_tot} Rows | {attr_uniq} Unique IDs")

# 2. Orphan Check (Cross-Table Verification)
# Are there users in Clickstream that don't exist in our core Attributes table?
orphans = df_click.select("Customer_ID").subtract(df_attr.select("Customer_ID")).count()
print(f"\nOrphaned IDs (Clicks with no profile): {orphans}")

# 3. ID Formatting & Missing Check
print("\nChecking ID Formatting (Attributes Table):")
df_attr.select(
    F.min(length("Customer_ID")).alias("Min_ID_Length"),
    F.max(length("Customer_ID")).alias("Max_ID_Length"),
    F.sum(F.when(col("Customer_ID").isNull() | (col("Customer_ID") == ""), 1).otherwise(0)).alias("Missing_IDs")
).show()

# Catch hidden whitespace in IDs
df_attr_spaces = df_attr.filter(length(col("Customer_ID")) != length(trim(col("Customer_ID")))).count()
print(f"IDs with hidden spaces: {df_attr_spaces}")

=== PHASE 1: FOUNDATION & KEY FORENSICS ===
Clickstream: 215376 Rows | 8974 Unique IDs
Financials:  12500 Rows | 12500 Unique IDs
Attributes:  12500 Rows | 12500 Unique IDs

Orphaned IDs (Clicks with no profile): 0

Checking ID Formatting (Attributes Table):
+-------------+-------------+-----------+
|Min_ID_Length|Max_ID_Length|Missing_IDs|
+-------------+-------------+-----------+
|            9|           10|          0|
+-------------+-------------+-----------+

IDs with hidden spaces: 0


In [8]:
print("=== PHASE 2: POISON & FAKE NULL DETECTION ===")

suspicious_strings = ["NM", "N/A", "NA", "null", "NULL", "-", "None", ""]

# 1. Hunt for Fake Nulls and Pure Whitespace
print("Hunting for Fake Nulls ('NM', 'N/A', etc.)...")
for c in df_fin.columns:
    fake_nulls = df_fin.filter(col(c).isin(suspicious_strings) | col(c).rlike(r"^\s+$")).count()
    if fake_nulls > 0:
        print(f"  [Financials] '{c}' has {fake_nulls} fake nulls.")

for c in df_attr.columns:
    fake_nulls = df_attr.filter(col(c).isin(suspicious_strings) | col(c).rlike(r"^\s+$")).count()
    if fake_nulls > 0:
        print(f"  [Attributes] '{c}' has {fake_nulls} fake nulls.")

# 2. Hunt for Underscores and Special Characters (Using robust regex)
print("\nHunting for Underscores & Illegal Characters...")
# Checking supposed numeric columns poisoned into strings
df_fin.filter(col("Annual_Income").rlike("_")).select("Customer_ID", "Annual_Income").show(3)
df_attr.filter(col("Age").rlike("_")).select("Customer_ID", "Age").show(3)

# 3. Check for overall missing values (Real Nulls)
print("\nReal Null Count (Financials):")
df_fin.select([F.sum(F.when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df_fin.columns]).show(vertical=True)

=== PHASE 2: POISON & FAKE NULL DETECTION ===
Hunting for Fake Nulls ('NM', 'N/A', etc.)...
  [Financials] 'Payment_of_Min_Amount' has 1438 fake nulls.

Hunting for Underscores & Illegal Characters...
+-----------+-------------+
|Customer_ID|Annual_Income|
+-----------+-------------+
| CUS_0x1009|    52312.68_|
| CUS_0x107c|    49718.55_|
| CUS_0x1098|    20652.98_|
+-----------+-------------+
only showing top 3 rows

+-----------+-----+
|Customer_ID|  Age|
+-----------+-----+
| CUS_0x1032|  40_|
| CUS_0x1057|  46_|
| CUS_0x10e7|3843_|
+-----------+-----+
only showing top 3 rows


Real Null Count (Financials):
-RECORD 0------------------------
 Customer_ID              | 0    
 Annual_Income            | 0    
 Monthly_Inhand_Salary    | 0    
 Num_Bank_Accounts        | 0    
 Num_Credit_Card          | 0    
 Interest_Rate            | 0    
 Num_of_Loan              | 0    
 Type_of_Loan             | 1426 
 Delay_from_due_date      | 0    
 Num_of_Delayed_Payment   | 0    
 Changed

In [9]:
print("=== PHASE 3: CATEGORICAL & TEXT PROFILING ===")

# 1. Find Text Concatenation Outliers (e.g., "DoctorDoctor" or "8669_")
print("Text Length Boundaries (Attributes):")
df_attr.select(
    F.min(length("Age")).alias("Min_Age_Len"),
    F.max(length("Age")).alias("Max_Age_Len"),
    F.min(length("Name")).alias("Min_Name_Len"),
    F.max(length("Name")).alias("Max_Name_Len"),
    F.max(length("Occupation")).alias("Max_Occ_Len")
).show()

# 2. Strict Distinct Value Checks
# This reveals typos and exact text formats (e.g., finding the 1426 missing loans)
print("\nExact Categories in 'Credit_Mix':")
df_fin.groupBy("Credit_Mix").count().orderBy("count").show()

print("\nExact Categories in 'Type_of_Loan' (Top 10):")
df_fin.groupBy("Type_of_Loan").count().orderBy(F.desc("count")).show(10, truncate=False)

print("\nExact Categories in 'Payment_Behaviour' (Checking for 'NM'):")
df_fin.groupBy("Payment_Behaviour").count().orderBy("count").show()

=== PHASE 3: CATEGORICAL & TEXT PROFILING ===
Text Length Boundaries (Attributes):
+-----------+-----------+------------+------------+-----------+
|Min_Age_Len|Max_Age_Len|Min_Name_Len|Max_Name_Len|Max_Occ_Len|
+-----------+-----------+------------+------------+-----------+
|          2|          5|           2|          25|         13|
+-----------+-----------+------------+------------+-----------+


Exact Categories in 'Credit_Mix':
+----------+-----+
|Credit_Mix|count|
+----------+-----+
|       Bad| 2360|
|         _| 2611|
|      Good| 3032|
|  Standard| 4497|
+----------+-----+


Exact Categories in 'Type_of_Loan' (Top 10):
+-----------------------+-----+
|Type_of_Loan           |count|
+-----------------------+-----+
|NULL                   |1426 |
|Not Specified          |176  |
|Credit-Builder Loan    |160  |
|Personal Loan          |159  |
|Debt Consolidation Loan|158  |
|Student Loan           |155  |
|Payday Loan            |150  |
|Mortgage Loan          |147  |
|Auto Loan

In [10]:
print("=== PHASE 4: REALITY & BOUNDARY CHECKS ===")

# Describe the surviving numeric columns to catch impossible minimums/maximums
numeric_cols = ["Monthly_Inhand_Salary", "Num_Bank_Accounts", "Interest_Rate", "Num_Credit_Card", "Delay_from_due_date", "Credit_Utilization_Ratio"]

df_fin.select(numeric_cols).describe().show()

=== PHASE 4: REALITY & BOUNDARY CHECKS ===
+-------+---------------------+------------------+------------------+------------------+-------------------+------------------------+
|summary|Monthly_Inhand_Salary| Num_Bank_Accounts|     Interest_Rate|   Num_Credit_Card|Delay_from_due_date|Credit_Utilization_Ratio|
+-------+---------------------+------------------+------------------+------------------+-------------------+------------------------+
|  count|                12500|             12500|             12500|             12500|              12500|                   12500|
|   mean|   4188.5923027131585|          16.93992|          73.21336|          23.17272|           21.06088|       32.34926457800977|
| stddev|   3180.1476109204173|114.35081529709485|468.68222710052765|132.00586642912222| 14.863091419503656|       5.156815304530768|
|    min|    303.6454166666666|                -1|                 1|                 0|                 -5|       20.10076996070649|
|    max|   15204.6

In [11]:
import pyspark
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder.appName("Validation").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Load ONE month of your final, cleaned Gold data
df_gold = spark.read.parquet("datamart/gold/feature_store/gold_feature_store_2023_01_01.parquet")

print("=== FINAL PIPELINE VALIDATION ===")

# 1. Did we fix the IDs? (Should be exactly 0)
malformed_ids = df_gold.filter(F.length("Customer_ID") != 10).count()
print(f"Malformed IDs Remaining: {malformed_ids}")

# 2. Did we fix the Underscores? (Should be exactly 0)
poisoned_ages = df_gold.filter(F.col("Age").cast("string").rlike("_")).count()
print(f"Underscores in Age Remaining: {poisoned_ages}")

# 3. Did we fix the Fake Nulls and Garbage?
print("\nCleaned Payment Behaviour Categories:")
df_gold.groupBy("Payment_Behaviour").count().show()

print("\nCleaned Type of Loan Categories:")
df_gold.groupBy("Type_of_Loan").count().show(truncate=False)

=== FINAL PIPELINE VALIDATION ===
Malformed IDs Remaining: 548
Underscores in Age Remaining: 0

Cleaned Payment Behaviour Categories:
+--------------------+-----+
|   Payment_Behaviour|count|
+--------------------+-----+
|Low_spent_Small_v...|  144|
|High_spent_Medium...|  106|
|                NULL| 8444|
|             Unknown|   38|
|High_spent_Small_...|   49|
|Low_spent_Large_v...|   45|
|Low_spent_Medium_...|   71|
|High_spent_Large_...|   77|
+--------------------+-----+


Cleaned Type of Loan Categories:
+----------------------------------------------------------------------------------------------------------------------------------+-----+
|Type_of_Loan                                                                                                                      |count|
+----------------------------------------------------------------------------------------------------------------------------------+-----+
|Personal Loan, Debt Consolidation Loan, Personal Loan, Home Equit